# NB_ingest_to_satellites — Silver → Satellites (append-only)

For each satellite defined in the model config:
1. Compute the hub hash key (HK) and diff hash (DIFF_HK) from tracked columns
2. LEFT JOIN against the latest DIFF_HK already stored in the satellite
3. INSERT only rows where DIFF_HK has changed (new row) or no previous row exists

Satellites are **append-only** — no updates or deletes, change detection via DIFF_HK.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
import uuid as _uuid

dbutils.widgets.text("CATALOG",       "workspace",  "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA",  "vault",      "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver",     "Silver schema")
dbutils.widgets.text("MODEL_PATH",    "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS",  "",           "Optional: only process records >= this timestamp")
dbutils.widgets.text("ALERT_CHANNEL", "log",        "log | webhook | all")
dbutils.widgets.text("WEBHOOK_URL",   "",           "Optional Slack/Teams webhook")

CATALOG        = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA   = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA  = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH     = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS   = dbutils.widgets.get("WATERMARK_TS")
ALERT_CHANNEL  = dbutils.widgets.get("ALERT_CHANNEL")
WEBHOOK_URL    = dbutils.widgets.get("WEBHOOK_URL") or None
VAULT_RUN_ID   = str(_uuid.uuid4())

model = load_model(MODEL_PATH)
print(f"Loaded {len(model['satellites'])} satellites from model")

In [ ]:
import uuid as _uuid

dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver", "Silver schema")
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS", "", "Optional: only process records >= this timestamp")

CATALOG       = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA  = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH    = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS  = dbutils.widgets.get("WATERMARK_TS")
VAULT_RUN_ID  = str(_uuid.uuid4())

model = load_model(MODEL_PATH)
print(f"Loaded {len(model['satellites'])} satellites from model")

In [ ]:
# ---------------------------------------------------------------------------
# Task 3.1 — Satellite DQ: no orphaned rows (HK must exist in parent hub)
# Task 4.3 — Alert on FAIL
# ---------------------------------------------------------------------------
print('\nSatellite ingestion complete.')
dq_failures = []

for sat_cfg in model['satellites']:
    if not sat_cfg.get('enabled', True):
        continue

    sat_name   = sat_cfg['name']
    parent_hub = sat_cfg['parent_hub']
    hk_col     = f"HK_{parent_hub[4:]}"
    sat_tbl    = f"{CATALOG}.{VAULT_SCHEMA}.{sat_name.lower()}"
    hub_tbl    = f"{CATALOG}.{VAULT_SCHEMA}.{parent_hub.lower()}"
    new_cnt    = sat_new_rows.get(sat_name, 0)
    print(f'  {sat_name}: {new_cnt:,} new rows')

    orphan_count = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM {sat_tbl} s
        LEFT JOIN {hub_tbl} h ON s.{hk_col} = h.{hk_col}
        WHERE h.{hk_col} IS NULL
    """).collect()[0]["n"] or 0

    status = "PASS" if orphan_count == 0 else "FAIL"
    write_dq_result(
        spark, run_id=VAULT_RUN_ID, layer="vault", table_name=sat_name.lower(),
        check_name="no_orphan_rows", status=status,
        observed_value=float(orphan_count), threshold=0.0,
        message=f"{orphan_count} orphaned row(s) in {sat_name}" if orphan_count else None,
    )
    if status == "FAIL":
        alert_dq_failure(
            table_name=sat_name.lower(), check_name="no_orphan_rows",
            observed_value=orphan_count, layer="vault",
            alert_channel=ALERT_CHANNEL, webhook_url=WEBHOOK_URL,
        )
        dq_failures.append(f"{sat_name}: {orphan_count} orphaned row(s)")

if dq_failures:
    raise RuntimeError("Satellite DQ FAIL:\n" + "\n".join(dq_failures))

print("Satellite DQ checks PASSED.")

In [ ]:
# ---------------------------------------------------------------------------
# Task 3.1 — Satellite DQ: no orphaned rows (HK must exist in parent hub)
# ---------------------------------------------------------------------------
print('\nSatellite ingestion complete.')
dq_failures = []

for sat_cfg in model['satellites']:
    if not sat_cfg.get('enabled', True):
        continue

    sat_name   = sat_cfg['name']
    parent_hub = sat_cfg['parent_hub']
    hk_col     = f"HK_{parent_hub[4:]}"
    sat_tbl    = f"{CATALOG}.{VAULT_SCHEMA}.{sat_name.lower()}"
    hub_tbl    = f"{CATALOG}.{VAULT_SCHEMA}.{parent_hub.lower()}"
    new_cnt    = sat_new_rows.get(sat_name, 0)
    print(f'  {sat_name}: {new_cnt:,} new rows')

    # Orphan check: satellite HK must exist in parent hub
    orphan_count = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM {sat_tbl} s
        LEFT JOIN {hub_tbl} h ON s.{hk_col} = h.{hk_col}
        WHERE h.{hk_col} IS NULL
    """).collect()[0]["n"] or 0

    status = "PASS" if orphan_count == 0 else "FAIL"
    write_dq_result(
        spark, run_id=VAULT_RUN_ID, layer="vault", table_name=sat_name.lower(),
        check_name="no_orphan_rows", status=status,
        observed_value=float(orphan_count), threshold=0.0,
        message=f"{orphan_count} orphaned row(s) in {sat_name}" if orphan_count else None,
    )
    if status == "FAIL":
        dq_failures.append(f"{sat_name}: {orphan_count} orphaned row(s)")

if dq_failures:
    raise RuntimeError("Satellite DQ FAIL:\n" + "\n".join(dq_failures))

print("Satellite DQ checks PASSED.")